<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Day 1 Lab3</h2>
            <span style="color:#ff7800;">OpenAI SDK Agent<br/>
            This is the third lab.<br/>
            The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

## And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is!

# This Lab Has Three Parts

## Part 1: Creating the agent

## Part 2: Tracing it

## Part 3: Adding Memory

So, different Agent calls will know about each other.

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

## Adding Observability with a trace

In [ ]:

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Wi-Fi")
    
print(result.final_output)

### side notes:
await in the code waits until the run is finished.<br>
You get one complete string: result.final_output.<br>
Nothing prints while the model is still generating.<br>
trace(...) only wraps this run for the OpenAI traces page.

## Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

### side notes:
You do not wait for the full answer first.<br>
Each event.data.delta is a small chunk of text (often a word or a few characters).<br>
print(..., end="", flush=True) writes those chunks in place, like ChatGPT typing.<br>
The loop filters to ResponseTextDeltaEvent so you ignore tool/other events and only show the assistant’s words.<br>


## Memory

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that. First, a quick reminder of what we saw in the last lab:

In [7]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Peter Pan.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Peter Pan.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
# adding the user's message to the conversation history

next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built-in SQLite session

In [12]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

### Sidenote:

"12346" is not a memory size. It is a session id: a name for one conversation.<br>

Where does it live? RAM<br>

How much memory? Whatever that conversation stores, until you restart the kernel.

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Peter Pan.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)